### 데이터 수집 + 청킹

In [24]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader, DirectoryLoader
from pathlib import Path
txt_loader = DirectoryLoader(
    './data',
    glob = '*.txt',
    loader_cls=TextLoader
)
txt_docs = txt_loader.load()
print(f'텍스트로더 완료, 문서 수: {len(txt_docs)}\n\
    doc0 일부: {txt_docs[0].page_content[:10]}...\n\
    doc0 일부: {txt_docs[1].page_content[:10]}...')

# txt 메타데이터 추가
for idx, doc in enumerate(txt_docs):
    filename = Path(doc.metadata['source']).name
    doc.metadata['filename'] = filename
    doc.metadata['file_type'] = 'txt'
    doc.metadata['doc_id'] = f'txt_{idx}'
    doc.metadata['category'] = 'RAG'


# text_path = 'data/sample1_text.txt'
# text_loader = TextLoader(text_path, encoding='utf-8')
# txt_docs = text_loader.load()
# print(f'텍스트로더 완료, 문서 수: {len(txt_docs)} 내용 일부: {txt_docs[0].page_content[:10]}...')

# pdf 파일
pdf_loader = DirectoryLoader(
    './data',
    glob = '*.pdf',
    loader_cls=PyPDFLoader
)
pdf_docs = pdf_loader.load()

# pdf 메타데이터 추가
for idx, doc in enumerate(pdf_docs):
    filename = Path(doc.metadata['source']).name
    doc.metadata['filename'] = filename
    doc.metadata['file_type'] = 'pdf'
    doc.metadata['doc_id'] = f'pdf_{idx}'
    doc.metadata['category'] = '규정/규칙'

print(f'텍스트로더 완료, 문서 수: {len(pdf_docs)} 내용 일부: {pdf_docs[0].page_content[:10]}...')

docs = txt_docs + pdf_docs

텍스트로더 완료, 문서 수: 2
    doc0 일부: LangGraph란...
    doc0 일부: 파인튜닝이란? Lo...
텍스트로더 완료, 문서 수: 35 내용 일부: [배경] 
안녕하세...


In [25]:
len(docs),len(pdf_docs), len(txt_docs)

(37, 35, 2)

In [26]:
# 웹페이지 크롤링 로드(Beatifulsoup 기반)
url = 'https://ko.wikipedia.org/wiki/AI'
web_loader = WebBaseLoader(url)
web_docs = web_loader.load()
print(f'웹페이지 로드 완료 제목: {web_docs[0].metadata.get("title", "N/A")} 내용일부: {web_docs[0].page_content[:20]}...')

웹페이지 로드 완료 제목: 인공지능 - 위키백과, 우리 모두의 백과사전 내용일부: 



인공지능 - 위키백과, 우리 ...


In [27]:
docs += web_docs
len(docs),len(pdf_docs), len(txt_docs), len(web_docs)

(38, 35, 2, 1)

#### 청킹(Chunking)
- 기본 청킹, 시멘틱 청킹
- 단순 글자수 기반 분할 Recursive...

In [28]:
# 가장 흔하게 사용
from langchain_text_splitters import RecursiveCharacterTextSplitter
basic_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 20,
    separators=['\n\n', '\n', '.', ' ', ',']
)
basic_chunks = basic_splitter.split_documents(docs)
print(f'[기본청킹] 분할된 청킹수 : {len(basic_chunks)}개')
for i,c in enumerate(basic_chunks[:4], 1):
    content = c.page_content.replace("\n", " ")
    print(f'chunk {i} : {content}... (길이: {len(c.page_content)}자)')

[기본청킹] 분할된 청킹수 : 946개
chunk 1 : LangGraph란 무엇인가요?... (길이: 17자)
chunk 2 : LangChain에서 만든 LangGraph는 복잡한 생성형 AI 에이전트 워크플로를 구축, 배포, 관리하도록 설계된 오픈 소스 AI 에이전트 프레임워크입니다... (길이: 88자)
chunk 3 : . 이는 사용자가 확장가능한 방식으로 대규모 언어 모델(LLM)을 만들어서 실행하고 최적화하게 해주는 툴과 라이브러리 세트를 제공합니다... (길이: 75자)
chunk 4 : . LangGraph의 핵심은 그래프 기반 아키텍처의 힘을 사용하여 AI 에이전트 워크플로의 다양한 구성 요소 간의 복잡한 관계를 모델링하고 관리하는 것입니다.... (길이: 89자)


#### 시멘틱 청킹 : 의미기반 분할
- 임베딩 모델, 코사인 유사도가 급격히 떨어지는 임계점(Threshold)에서 텍스트를 자른다


In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# 임베딩 모델
embeddings = HuggingFaceEmbeddings(model_name = 'jhgan/ko-sroberta-multitask')

# 시멘틱 청커
semantic_splitter = SemanticChunker(embeddings, breakpoint_threshold_type='percentile') # 문맥 변화량이 상위 N%인 지점에서 절단

semantic_chunks = semantic_splitter.split_documents(docs)

print(f'[시멘틱청킹] 분할된 청킹수 : {len(semantic_chunks)}개')
for i,c in enumerate(semantic_chunks[-4:], 1):
    content = c.page_content.replace("\n", " ")
    print(f'Semantic chunk {i} : {content}... (길이: {len(c.page_content)}자)')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9893.40it/s]


[시멘틱청킹] 분할된 청킹수 : 119개
Semantic chunk 1 : The Guardian. 외부 링크  위키미디어 공용에 인공지능 관련 미디어 분류가 있습니다. 〈인공지능〉. 《두산세계대백과사전》. (주)두산. 영화 속의 인공지능 5부작 - 인공지능을 다룬 명작 영화들을 소개하며, 주제별로 묶어 현재와 미래에 대한 질문을 던진다. AI Study — 한국 최대의 인공지능 정보 구축 사이트 KAIST 인공지능 연구실 - KAIST 인공지능 연구실로 다양한 연구 과제와 진행 상태를 볼 수 있다. 한겨레21 세계 최초? 가장 위험한 선례가 될 수도 있다 2026-03-27 vte기술 기술의 개요 응용과학의 개요 분야농업 농공학 수산양식학 수산학 식품화학 식품공학 식품미생물학 식품 기술 GURT ICT 영양학 생물의학 생물정보학 생물공학 바이오메카트로닉스 의공학 생명공학기술 화학정보학 유전공학 보건 의학 연구 보건의료기술 나노의학 신경과학 신경 기술 약리학 생식 기술 조직공학 건축과 건설 음향공학 건축공학 건물 서비스 공학 토목공학 건설공학 국내 기술 퍼사이드 공학 화재방지공학 안전공학 위생공학 구조공학 교육 교육용 소프트웨어 교육공학 교육에서의 ICT 영향 교육공학 가상 캠퍼스 에너지 원자력공학 원자력 기술 석유공학 소프트 에너지 기술 환경 청정 에너지 그린콜 환경 디자인 생태공학 에코기술 환경공학 환경공학과학 그린 빌딩 녹색 나노기술 경관공학 재생 가능 에너지 지속 가능한 디자인 지속 가능한 공학 산업 자동화 비즈니스 인포매틱스 공학 관리 기업공학 금융공학 생명공학기술 산업공학 금속공학 채굴공학 생산성 연구개발 마찰공학 IT와 통신 인공지능 방송공학 컴퓨터 공학 컴퓨터 과학 핀테크 정보기술 음악 기술 온톨로지 엔지니어링 RF 엔지니어링 소프트웨어 공학 통신공학 시각공학 웹 공학 군사 육군 공학 정비 전자전 군 통신기술 공병 스텔스 기술 교통 항공우주공학 자동차 공학 선박공학 우주 기술 트래픽 공학 교통공학 기타응용과학 저온학 전기광학 토목지질학 기초공학 

#### 효율적 검색을 위한 메타데이터 설계
- 텍스트(context)를 DB에 넣으면 어떤 파일의 몇 번째 단락인지, 언제 수집된 데이터인지 등... 알 수 없음
- 이를 방지하기 위해 chunking단계에서 정밀하게 추가해야 함

In [30]:
import datetime
for i, chunk in enumerate(semantic_chunks):
    # 기존 문서의 source 속성에 추가로 인댁스와 시간정보 추가
    chunk.metadata['chunk_index'] = i
    chunk.metadata['timepstamp'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    # chunk.metadata['category'] = 'InnerDocs'

for key,value in semantic_chunks[0].metadata.items():
    print(key, value)


source data\sample1_text.txt
filename sample1_text.txt
file_type txt
doc_id txt_0
category RAG
chunk_index 0
timepstamp 2026-06-05 12:49:14


In [31]:
semantic_chunks

[Document(metadata={'source': 'data\\sample1_text.txt', 'filename': 'sample1_text.txt', 'file_type': 'txt', 'doc_id': 'txt_0', 'category': 'RAG', 'chunk_index': 0, 'timepstamp': '2026-06-05 12:49:14'}, page_content='LangGraph란 무엇인가요? LangChain에서 만든 LangGraph는 복잡한 생성형 AI 에이전트 워크플로를 구축, 배포, 관리하도록 설계된 오픈 소스 AI 에이전트 프레임워크입니다. 이는 사용자가 확장가능한 방식으로 대규모 언어 모델(LLM)을 만들어서 실행하고 최적화하게 해주는 툴과 라이브러리 세트를 제공합니다. LangGraph의 핵심은 그래프 기반 아키텍처의 힘을 사용하여 AI 에이전트 워크플로의 다양한 구성 요소 간의 복잡한 관계를 모델링하고 관리하는 것입니다. 이 모든 정보는 무엇을 의미할까요? 다음 예제를 보면 LangGraph를 더 명확하게 이해할 수 있습니다. 이 그래프 기반 아키텍처를 강력하고 구성 가능한 "슈퍼 지도"라고 생각해 보세요. 사용자는 AI 워크플로가 이 "슈퍼 지도"의 "내비게이터"로 생각할 수 있습니다. 마지막으로, 이 예시에서 사용자는 "지도 제작자"입니다. 이런 의미에서 내비게이터는 "지도 제작자"가 만든 "슈퍼 지도"에서 두 지점을 잇는 최적의 경로를 차트로 표시합니다. 요약하자면, 그래프 기반 아키텍처("수퍼 지도") 속 최적의 경로는 AI 워크플로("Navigator")를 사용하여 차트로 작성하고 탐색합니다. 이 비유를 통해 LangGraph를 이해해 보세요. 지도를 좋아하신다면 지도 제작자라는 단어가 사용되는 것도 보실 수 있는 기회입니다. LangGraph 워크플로\nLangGraph는 AI 워크플로 안에서 이루어지는 프로세스를 조명하여 에이전트 상태를 완전히 투명하게 공개합니다. LangGraph에 있는 "상태" 기능은 AI

### 임베딩 + ChromaDB 구축

#### 왜 단순 검색(Similarity) 대신 MMR 검색을 활용하는가? (설계 핵심 방향)

Vector DB에 문서가 저장된 후, 사용자의 질문에 답변하기 위해 가장 유사한 문서를 찾아옵니다(Similarity Search). 하지만 유사도에만 집착하면 "모두 똑같은 소리만 하는" 중복된 문맥들만 검색될 위험이 있습니다.

MMR(Maximal Marginal Relevance) 알고리즘 은 문서와 질문 간의 '유사도'를 챙기면서도, 검색된 문서들 간의 '다양성(Diversity)'을 동시에 고려합니다. 이를 통해 LLM에게 편향되지 않고 다각적인 정보를 제공하여, 더욱 풍부하고 정확한 답변을 유도(환각 방지)하는 핵심 기술입니다.


In [32]:
from langchain_chroma import Chroma
# 디렉터리 설정
db_directory = "./chroma_db_session"

vector_db = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    persist_directory=db_directory
)

#### 검색 방법론 비교... similarity search VS MMR search

In [33]:
query = "langgraph에 대해서 알려주세요"
print('일반 유사도 기반 검색 결과')
# 검색의 다양성을 위해 처음에 가져올 문서 수
# lambda_mult : 1에 가까울수록 유사도 중시, 0에 가까울수록 다양성 중시, 보통 0.5
sim_results = vector_db.max_marginal_relevance_search(query, k=2, fetch_k =5, lambda_mult=0.5)
for idx, doc in enumerate(sim_results):
    print(f' - {idx} {doc.page_content.strip()}...')

일반 유사도 기반 검색 결과
 - 0 또한 이러한 프로세스를 처음 접하거나 무대 뒤에서 무슨 일이 일어나는지 알고 싶어 하는 개인의 참여도를 높일 수 있습니다. LangGraph는 AI 애플리케이션을 구축하기 위한 Python 프레임워크인 LangChain을 비롯해 여러 핵심 기술을 기반으로 구축되었습니다. LangChain에는 LLM을 구축하고 관리하기 위한 라이브러리가 포함되어 있습니다. 그리고 LangGraph는 휴먼인더루프 접근 방식을 사용합니다. 이 기술들을 API 및 툴 세트와 결합하여 사용자에게 챗봇, 상태 그래프, 기타 에이전트 기반 시스템을 포함한 AI 솔루션과 워크플로를 개발할 수 있는 다목적 플랫폼을 제공합니다. LangGraph의 주요 기능, 이점, 사용 사례를 알아보면서 LangGraph의 세계를 더 깊이 탐구하세요....
 - 1 22, Russell & Norvig 2003, pp. 949–950, Hofstadter 1980, pp. 471–477 and see Lucas 1961
↑ "Know-how" is Dreyfus' term. (Dreyfus makes a distinction between "knowing how" and "knowing that", a modern version of Heidegger's distinction of ready-to-hand and present-at-hand.) (Dreyfus & Dreyfus 1986)
↑ Dreyfus' critique of artificial intelligence: McCorduck 2004, pp. 211–239, Crevier 1993, pp. 120–132, Russell & Norvig 2003, pp. 950–952 and see Dreyfus 1965, Dreyfus 1972, Dreyfus & Dreyfus 1986
↑ Searle's critique of AI: McCorduck 2004, pp. 443–445, Crevier 1993, pp. 26

#### 프롬프트 엔지니어링

In [34]:
import os
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate
# 환각방지 지시문 설계
system_template = '''당신은 친절하고 정확한 어시스턴트입니다.
반드시 아래의 [제공된 문서]만을 바탕으로 사용자의 질문에 답변하세요
만약 [제공된 문서]에 질문에 대한 명확한 답변이 없다면, 추론하지 말고 "과련된 정보가 문서에 없습니다." 라고 답하세요

[제공된 문서]
{context}
'''

human_template = "질문: {question}"
chat_prompt = ChatPromptTemplate.from_messages([
    ('system', system_template), ('human',human_template)
])
formatted_prompt = chat_prompt.format(
    context = 'RAG는 검색 증각 생성의 약자입니다. 기업 내부 문서를 기반으로 AI가 답변하게 만듭니다.',
    question = 'LangGraph란 무엇인가요?'
)

formatted_prompt

'System: 당신은 친절하고 정확한 어시스턴트입니다.\n반드시 아래의 [제공된 문서]만을 바탕으로 사용자의 질문에 답변하세요\n만약 [제공된 문서]에 질문에 대한 명확한 답변이 없다면, 추론하지 말고 "과련된 정보가 문서에 없습니다." 라고 답하세요\n\n[제공된 문서]\nRAG는 검색 증각 생성의 약자입니다. 기업 내부 문서를 기반으로 AI가 답변하게 만듭니다.\n\nHuman: 질문: LangGraph란 무엇인가요?'

In [ ]:
# few-shot 프롬프트를 통한 응답 강화
